In [1]:
%pip install PyMuPDF requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import requests
import os
import re
from typing import List, Dict, Any, Optional

In [3]:
# ==============================================================================
# PARTIE I : Gestion de l'API (AiClient)
# ==============================================================================

class AiClient:
    """
    Client pour interagir avec une API de modèle de langage (similaire à OpenRouter/OpenAI).
    Gère les requêtes HTTP POST et l'authentification.
    """
    def __init__(self, api_endpoint: str, api_key: str, default_model: str, max_tokens: int = 4000):
        self.api_endpoint = api_endpoint
        self.api_key = api_key
        self.default_model = default_model
        self.max_tokens = max_tokens  # Limite de contexte pour Mistral-7B

    def call_with_prompt(self, prompt: str, system_prompt: str = "", model: str = None, max_tokens: int = 1000) -> Optional[Dict[str, Any]]:
        """
        Appelle l'API avec un prompt utilisateur et un prompt système optionnel.
        """
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }

        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})

        messages.append({"role": "user", "content": prompt})

        payload = json.dumps({
            "model": model if model else self.default_model,
            "messages": messages,
            "temperature": 0.2,
            "max_tokens": max_tokens,
        })

        try:
            response = requests.post(
                url=self.api_endpoint,
                headers=headers,
                data=payload,
                timeout=60
            )
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Erreur lors de l'appel API: {e}")
            return None

    def estimate_tokens(self, text: str) -> int:
        """Estimation approximative du nombre de tokens (1 token ≈ 4 caractères)."""
        return len(text) // 4

In [4]:
# ==============================================================================
# PARTIE II : Extraction et Découpage du PDF
# ==============================================================================

def extract_text_from_pdf(file_path: str) -> str:
    """
    Extrait le texte brut d'un fichier PDF.
    Nécessite: pip install PyMuPDF (fitz)
    """
    try:
        import fitz  # PyMuPDF

        print(f"--- EXTRACTION DU FICHIER PDF: {file_path} ---")
        doc = fitz.open(file_path)
        full_text = ""

        for page_num in range(len(doc)):
            page = doc[page_num]
            full_text += page.get_text()

        doc.close()
        print(f"✓ Extraction réussie: {len(full_text)} caractères extraits")
        return full_text

    except ImportError:
        print("ERREUR: La librairie PyMuPDF (fitz) n'est pas installée.")
        print("Installation: pip install PyMuPDF")
        return ""
    except Exception as e:
        print(f"ERREUR lors de l'extraction du PDF: {e}")
        return ""


def segment_report_by_sections(full_text: str) -> Dict[str, str]:
    """
    Découpe le texte du rapport en sections basées sur les titres.
    Retourne un dictionnaire {titre_section: contenu}.
    """
    print("--- DÉCOUPAGE DU RAPPORT EN SECTIONS ---")

    # Patterns de regex pour identifier les sections principales
    section_patterns = [
        (r'Partie\s+I+\s*:?\s*Introduction\s+et\s+Contexte', 'Partie I: Introduction et Contexte'),
        (r'Partie\s+II\s*:?\s*Étude\s+Préliminaire', 'Partie II: Étude Préliminaire'),
        (r'II\.\s*Analyse\s+et\s+Conception', 'Partie II: Analyse et Conception'),
        (r'Partie\s+III\s*:?\s*Réalisation', 'Partie III: Réalisation'),
        (r'Partie\s+IV\s*:?\s*Discussion\s+et\s+Conclusion', 'Partie IV: Discussion et Conclusion'),
    ]

    sections = {}

    # Trouver les positions de chaque section
    section_positions = []
    for pattern, title in section_patterns:
        match = re.search(pattern, full_text, re.IGNORECASE)
        if match:
            section_positions.append((match.start(), title))

    # Trier par position dans le texte
    section_positions.sort(key=lambda x: x[0])

    # Extraire le contenu de chaque section
    for i, (start_pos, title) in enumerate(section_positions):
        # Déterminer la fin de la section (début de la section suivante ou fin du texte)
        if i + 1 < len(section_positions):
            end_pos = section_positions[i + 1][0]
        else:
            end_pos = len(full_text)

        # Extraire le contenu et nettoyer
        content = full_text[start_pos:end_pos].strip()
        sections[title] = content
        print(f"✓ Section '{title}': {len(content)} caractères")

    return sections


def chunk_large_section(section_content: str, max_chars: int = 8000) -> List[str]:
    """
    Découpe une section trop grande en morceaux plus petits.
    Utile pour respecter les limites de tokens de l'API.
    """
    if len(section_content) <= max_chars:
        return [section_content]

    chunks = []
    paragraphs = section_content.split('\n\n')
    current_chunk = ""

    for para in paragraphs:
        if len(current_chunk) + len(para) <= max_chars:
            current_chunk += para + "\n\n"
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = para + "\n\n"

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


def read_and_segment_report_file(file_path: str) -> Dict[str, str]:
    """
    Fonction principale pour lire un rapport PDF et le découper en sections.
    """
    # 1. Extraire le texte du PDF
    full_text = extract_text_from_pdf(file_path)

    if not full_text:
        print("ERREUR: Impossible d'extraire le texte du PDF.")
        return {}

    # 2. Découper en sections
    sections = segment_report_by_sections(full_text)

    if not sections:
        print("AVERTISSEMENT: Aucune section identifiée. Utilisation du texte complet.")
        sections = {'Rapport Complet': full_text}

    print(f"--- FIN DE L'EXTRACTION: {len(sections)} sections identifiées ---\n")
    return sections


In [5]:
# ==============================================================================
# PARTIE III : Chaîne de Prompts pour l'Évaluation
# ==============================================================================

def create_concise_prompt(section_content: str, instruction: str, max_chars: int = 6000) -> str:
    """
    Crée un prompt concis en limitant la taille du contenu de section.
    """
    if len(section_content) > max_chars:
        section_content = section_content[:max_chars] + "\n\n[... CONTENU TRONQUÉ POUR RESPECTER LES LIMITES DE TOKENS ...]"

    return f"""
{instruction}

CONTENU DE LA SECTION À ANALYSER (limité pour optimisation):
---
{section_content}
---
"""


def generate_detailed_feedback(report_sections: Dict[str, str], ai_client: AiClient) -> List[Dict[str, Any]]:
    """
    Génère des feedbacks détaillés et objectifs pour chaque section du rapport.
    Identifie les points forts, les lacunes et propose des améliorations concrètes.
    """
    feedback_results = []

    system_prompt_feedback = (
        "Vous êtes un expert pédagogique spécialisé dans l'accompagnement des étudiants. "
        "Votre rôle est de fournir des feedbacks CONSTRUCTIFS et ACTIONNABLES. "
        "Pour chaque section, identifiez: "
        "1) Ce qui est bien fait (points forts), "
        "2) Ce qui manque ou peut être amélioré (lacunes), "
        "3) Des recommandations concrètes et spécifiques (max 150 mots total)."
    )

    # Définition des critères d'évaluation par section
    section_criteria = {
        'Partie I: Introduction et Contexte': {
            'title': 'Introduction et Contexte',
            'criteria': """
Critères d'évaluation:
- Problématique: Claire, pertinente, bien délimitée?
- Objectifs: SMART (Spécifiques, Mesurables, Atteignables, Réalistes, Temporels)?
- Contexte: Justification du projet et enjeux bien exposés?
- Méthodologie: Approche clairement présentée?
            """
        },
        'Partie II: Étude Préliminaire': {
            'title': 'État de l\'Art et Choix Techniques',
            'criteria': """
Critères d'évaluation:
- Revue de littérature: Références pertinentes et à jour?
- Analyse comparative: Solutions existantes bien étudiées?
- Justification des choix: Technologies justifiées par rapport aux besoins?
- Veille technologique: Démonstration d'une connaissance du domaine?
            """
        },
        'Partie II: Analyse et Conception': {
            'title': 'Analyse et Conception',
            'criteria': """
Critères d'évaluation:
- Cahier des charges: Besoins fonctionnels/non-fonctionnels complets?
- Modélisation UML: Diagrammes pertinents, cohérents et complets?
- Architecture: Choix d'architecture justifié et adapté?
- Modèle de données: Structure claire et normalisée?
            """
        },
        'Partie III: Réalisation': {
            'title': 'Réalisation et Tests',
            'criteria': """
Critères d'évaluation:
- Implémentation: Fonctionnalités principales développées?
- Qualité du code: Bonnes pratiques respectées?
- Tests: Stratégie de test complète (unitaires, intégration, fonctionnels)?
- Défis techniques: Solutions apportées aux problèmes rencontrés?
            """
        },
        'Partie IV: Discussion et Conclusion': {
            'title': 'Discussion et Conclusion',
            'criteria': """
Critères d'évaluation:
- Analyse critique: Recul et auto-évaluation du projet?
- Limites: Identification honnête des contraintes rencontrées?
- Perspectives: Évolutions futures réalistes et pertinentes?
- Bilan des compétences: Apprentissages clairement identifiés?
            """
        }
    }

    print("\n🔍 GÉNÉRATION DE FEEDBACKS DÉTAILLÉS PAR SECTION")
    print("="*70)

    for section_key, section_content in report_sections.items():
        if section_key not in section_criteria:
            continue  # Ignorer les sections non définies

        criteria_info = section_criteria[section_key]

        print(f"\n📝 Analyse de: {criteria_info['title']}...")

        # Création du prompt de feedback
        prompt_feedback = f"""
Analysez la section suivante et fournissez un feedback structuré selon ce format:

✅ POINTS FORTS (2-3 éléments concrets)
❌ LACUNES/AXES D'AMÉLIORATION (2-3 éléments spécifiques)
💡 RECOMMANDATIONS CONCRÈTES (2-3 actions précises à entreprendre)

{criteria_info['criteria']}

SECTION À ANALYSER:
---
{section_content[:5000]}
---

Soyez précis et donnez des exemples concrets. Max 150 mots.
"""

        response = ai_client.call_with_prompt(
            prompt_feedback,
            system_prompt=system_prompt_feedback,
            max_tokens=400
        )

        if response:
            feedback_content = response["choices"][0]["message"]["content"]
            feedback_results.append({
                "section": criteria_info['title'],
                "feedback": feedback_content
            })
            print(f"✓ Feedback généré pour: {criteria_info['title']}")
        else:
            print(f"❌ Échec du feedback pour: {criteria_info['title']}")

    return feedback_results


def run_prompt_chain(report_sections: Dict[str, str], ai_client: AiClient) -> List[Dict[str, Any]]:
    """
    Exécute une chaîne de 5 prompts optimisée pour évaluer un rapport.
    Gère les limitations de tokens de Mistral-7B.
    """
    evaluation_results = []

    system_prompt = (
        "Vous êtes un évaluateur académique rigoureux. "
        "Analysez le texte fourni et fournissez des évaluations objectives et CONCISES. "
        "Limitez vos réponses à l'essentiel (max 150 mots par analyse)."
    )

    # Vérification des sections
    section_keys = [
        'Partie I: Introduction et Contexte',
        'Partie II: Étude Préliminaire',
        'Partie II: Analyse et Conception',
        'Partie III: Réalisation',
        'Partie IV: Discussion et Conclusion'
    ]

    for key in section_keys:
        if key not in report_sections:
            report_sections[key] = f"SECTION MANQUANTE: {key}"

    # --- ÉTAPE 1: Problématique et Objectifs ---
    section_1 = report_sections['Partie I: Introduction et Contexte']

    prompt1 = create_concise_prompt(
        section_1,
        "Évaluez la Problématique et les Objectifs (max 100 mots).\n"
        "Note sur 10 + analyse concise: problématique claire? Objectifs SMART?"
    )

    response1 = ai_client.call_with_prompt(prompt1, system_prompt=system_prompt, max_tokens=300)
    if not response1:
        print("❌ Échec à l'étape 1")
        return evaluation_results

    summary1 = response1["choices"][0]["message"]["content"]
    evaluation_results.append({"step": "1. Problématique & Objectifs", "result": summary1})
    print("✓ Étape 1 complétée")

    # --- ÉTAPE 2: État de l'Art ---
    section_2 = report_sections['Partie II: Étude Préliminaire']

    prompt2 = create_concise_prompt(
        section_2,
        f"Évaluez l'État de l'Art et Choix Techniques (max 100 mots).\n"
        f"Contexte précédent: {summary1[:200]}\n"
        f"État de l'art suffisant? Technologies justifiées?"
    )

    response2 = ai_client.call_with_prompt(prompt2, system_prompt=system_prompt, max_tokens=300)
    if not response2:
        print("❌ Échec à l'étape 2")
        return evaluation_results

    summary2 = response2["choices"][0]["message"]["content"]
    evaluation_results.append({"step": "2. État de l'Art & Choix Techniques", "result": summary2})
    print("✓ Étape 2 complétée")

    # --- ÉTAPE 3: Conception ---
    section_3 = report_sections['Partie II: Analyse et Conception']

    prompt3 = create_concise_prompt(
        section_3,
        "Évaluez la Conception UML et Modèle de données (max 100 mots).\n"
        "Note sur 10 + analyse: modèles complets et cohérents?"
    )

    response3 = ai_client.call_with_prompt(prompt3, system_prompt=system_prompt, max_tokens=300)
    if not response3:
        print("❌ Échec à l'étape 3")
        return evaluation_results

    summary3 = response3["choices"][0]["message"]["content"]
    evaluation_results.append({"step": "3. Qualité de la Conception", "result": summary3})
    print("✓ Étape 3 complétée")

    # --- ÉTAPE 4: Réalisation ---
    section_4 = report_sections['Partie III: Réalisation']

    prompt4 = create_concise_prompt(
        section_4,
        "Évaluez la Réalisation et Tests (max 100 mots).\n"
        "Défis techniques relevés? Stratégie de test adéquate?"
    )

    response4 = ai_client.call_with_prompt(prompt4, system_prompt=system_prompt, max_tokens=300)
    if not response4:
        print("❌ Échec à l'étape 4")
        return evaluation_results

    summary4 = response4["choices"][0]["message"]["content"]
    evaluation_results.append({"step": "4. Réalisation & Tests", "result": summary4})
    print("✓ Étape 4 complétée")

    # --- ÉTAPE 5: Synthèse Finale ---
    section_5 = report_sections['Partie IV: Discussion et Conclusion']

    # Synthèse des évaluations précédentes (limitée)
    previous_summary = f"""
1. {summary1[:100]}
2. {summary2[:100]}
3. {summary3[:100]}
4. {summary4[:100]}
"""

    prompt5 = f"""
Synthèse finale (MAX 150 mots).

Évaluations précédentes (résumé):
{previous_summary}

Section Conclusion:
{section_5[:4000]}

Rédigez un feedback final identifiant:
- LE point fort majeur
- LA principale zone d'amélioration
- Note globale suggérée sur 20
"""

    response5 = ai_client.call_with_prompt(prompt5, system_prompt=system_prompt, max_tokens=400)
    if not response5:
        print("❌ Échec à l'étape 5")
        return evaluation_results

    summary5 = response5["choices"][0]["message"]["content"]
    evaluation_results.append({"step": "5. Synthèse & Feedback Final", "result": summary5})
    print("✓ Étape 5 complétée")

    return evaluation_results


In [6]:
# def get_api_key() -> str:
#     """Récupère la clé API depuis les variables d'environnement ou fichier .env"""
#     try:
#         with open('.env', 'r') as file:
#             for line in file:
#                 if 'OPENROUTER_KEY' in line:
#                     return line.split('=')[1].strip().strip('"')
#     except FileNotFoundError:
#         pass

#     return os.environ.get("OPENROUTER_KEY", "VOTRE_CLE_API_ICI")

def get_api_key():
    try:
        with open('.env', 'r') as file:
            content = file.read().strip()
            # Handle different .env formats
            for line in content.split('\n'):
                if line.startswith('OPENROUTER_API_KEY'):
                    return line.split('=')[1].strip().strip('"').strip("'")
            # Fallback to first line
            return content.split('\n')[0].split('=')[1].strip().strip('"').strip("'")
    except FileNotFoundError:
        print("Error: .env file not found")
        sys.exit(1)
    except Exception as e:
        print(f"Error reading API key: {e}")
        sys.exit(1)

In [11]:
def main():
    """
    Fonction principale orchestrant l'analyse automatisée du rapport PDF.
    """
    print("="*70)
    print(" SYSTÈME D'ÉVALUATION AUTOMATISÉE DE RAPPORTS DE FIN DE FORMATION")
    print("="*70)

    # 1. Configuration de l'API
    openrouter_api = "https://openrouter.ai/api/v1/chat/completions"
    openrouter_key = get_api_key()

    # Initialisation du client IA avec limite de tokens pour Mistral-7B
    ai_client = AiClient(
        openrouter_api,
        openrouter_key,
        "mistralai/mistral-7b-instruct:free",
        max_tokens=8000  # Limite de contexte Mistral-7B
    )

    # 2. Chemin du fichier PDF
    report_file_path = "doc.pdf"

    if not os.path.exists(report_file_path):
        print(f"\n❌ ERREUR: Le fichier '{report_file_path}' n'existe pas.")
        print("Assurez-vous que le fichier PDF est dans le même répertoire que ce script.")
        return

    # 3. Extraction et Découpage du PDF
    print(f"\n📄 Lecture du fichier: {report_file_path}")
    report_sections = read_and_segment_report_file(report_file_path)

    if not report_sections:
        print("\n❌ ERREUR: Aucune section extraite. Vérifiez le format du PDF.")
        return

    # Afficher les sections détectées
    print("\n📋 Sections détectées:")
    for i, (title, content) in enumerate(report_sections.items(), 1):
        tokens = ai_client.estimate_tokens(content)
        print(f"  {i}. {title}: {len(content)} caractères (~{tokens} tokens)")

    # 4. Analyse par chaîne de prompts
    print("\n" + "="*70)
    print(" DÉMARRAGE DE L'ANALYSE AUTOMATISÉE PAR IA")
    print("="*70)

    results = run_prompt_chain(report_sections, ai_client)

    if not results:
        print("\n❌ L'analyse a échoué. Vérifiez votre connexion API et votre clé.")
        return

    # 5. Affichage des résultats d'évaluation
    print("\n" + "="*70)
    print(" RAPPORT D'ÉVALUATION FINALE")
    print("="*70)

    for i, res in enumerate(results, 1):
        print(f"\n{'─'*70}")
        print(f"ÉTAPE {i}: {res['step']}")
        print(f"{'─'*70}")
        print(res['result'])

    print("\n" + "="*70)
    print(" ANALYSE TERMINÉE AVEC SUCCÈS ✓")
    print("="*70)

    # 6. Génération des feedbacks détaillés par section
    print("\n" + "="*70)
    print(" GÉNÉRATION DES FEEDBACKS DÉTAILLÉS")
    print("="*70)

    feedbacks = generate_detailed_feedback(report_sections, ai_client)

    if feedbacks:
        print("\n" + "="*70)
        print(" FEEDBACKS OBJECTIFS POUR AMÉLIORATION")
        print("="*70)

        for i, feedback in enumerate(feedbacks, 1):
            print(f"\n{'═'*70}")
            print(f"📋 SECTION {i}: {feedback['section']}")
            print(f"{'═'*70}")
            print(feedback['feedback'])

        print("\n" + "="*70)
        print(" FEEDBACKS GÉNÉRÉS AVEC SUCCÈS ✓")
        print("="*70)

    # 7. Sauvegarde des résultats (évaluation + feedbacks)
    try:
        complete_report = {
            "evaluation": results,
            "feedbacks": feedbacks,
            "metadata": {
                "file": report_file_path,
                "sections_analyzed": len(report_sections),
                "total_chars": sum(len(content) for content in report_sections.values())
            }
        }

        with open('evaluation_complete.json', 'w', encoding='utf-8') as f:
            json.dump(complete_report, f, ensure_ascii=False, indent=2)
        print("\n💾 Rapport complet sauvegardé dans: evaluation_complete.json")

        # Sauvegarde aussi en format texte lisible
        with open('evaluation_complete.txt', 'w', encoding='utf-8') as f:
            f.write("="*70 + "\n")
            f.write(" RAPPORT D'ÉVALUATION COMPLÈTE\n")
            f.write("="*70 + "\n\n")

            f.write("## ÉVALUATIONS PAR ÉTAPES ##\n\n")
            for i, res in enumerate(results, 1):
                f.write(f"\nÉTAPE {i}: {res['step']}\n")
                f.write("─"*70 + "\n")
                f.write(res['result'] + "\n")

            f.write("\n\n" + "="*70 + "\n")
            f.write("## FEEDBACKS DÉTAILLÉS PAR SECTION ##\n")
            f.write("="*70 + "\n\n")

            for i, feedback in enumerate(feedbacks, 1):
                f.write(f"\nSECTION {i}: {feedback['section']}\n")
                f.write("═"*70 + "\n")
                f.write(feedback['feedback'] + "\n\n")

        print("💾 Version texte sauvegardée dans: evaluation_complete.txt")

    except Exception as e:
        print(f"\n⚠️ Impossible de sauvegarder les résultats: {e}")


if __name__ == "__main__":
    main()

 SYSTÈME D'ÉVALUATION AUTOMATISÉE DE RAPPORTS DE FIN DE FORMATION

📄 Lecture du fichier: doc.pdf
--- EXTRACTION DU FICHIER PDF: doc.pdf ---
✓ Extraction réussie: 32436 caractères extraits
--- DÉCOUPAGE DU RAPPORT EN SECTIONS ---
✓ Section 'Partie II: Analyse et Conception': 15192 caractères
--- FIN DE L'EXTRACTION: 1 sections identifiées ---


📋 Sections détectées:
  1. Partie II: Analyse et Conception: 15192 caractères (~3798 tokens)

 DÉMARRAGE DE L'ANALYSE AUTOMATISÉE PAR IA
✓ Étape 1 complétée
✓ Étape 2 complétée
✓ Étape 3 complétée
✓ Étape 4 complétée
✓ Étape 5 complétée

 RAPPORT D'ÉVALUATION FINALE

──────────────────────────────────────────────────────────────────────
ÉTAPE 1: 1. Problématique & Objectifs
──────────────────────────────────────────────────────────────────────
 L'analyse ne peut pas être effectuée car la section demandée (Introduction et Contexte) est manquante. Pour une évaluation objective, veuillez fournir le texte complet de la section "Introduction et Contex